# 3.5: Normalization from scratch

The chapter's suggested code lab implements all three layers and checks them against PyTorch. Here is a compact version:


In [ ]:
import torch
import torch.nn as nn

def batch_norm(x, gamma, beta, eps=1e-5):
    # x: (B, d); statistics over the batch dimension (training mode)
    mu = x.mean(dim=0, keepdim=True)
    var = x.var(dim=0, unbiased=False, keepdim=True)
    return gamma * (x - mu) / torch.sqrt(var + eps) + beta

def layer_norm(x, gamma, beta, eps=1e-5):
    # statistics over the feature dimension, per example
    mu = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, unbiased=False, keepdim=True)
    return gamma * (x - mu) / torch.sqrt(var + eps) + beta

def rms_norm(x, gamma, eps=1e-6):
    rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)
    return gamma * x / rms

B, d = 8, 16
x = torch.randn(B, d) * 3 + 1
g, b = torch.ones(d), torch.zeros(d)

print(torch.allclose(layer_norm(x, g, b), nn.LayerNorm(d)(x), atol=1e-5))
print(torch.allclose(batch_norm(x, g, b), nn.BatchNorm1d(d).train()(x), atol=1e-5))
print(torch.allclose(rms_norm(x, g), nn.RMSNorm(d, eps=1e-6)(x), atol=1e-5))

# BatchNorm output for example 0 depends on the rest of the batch; LayerNorm's does not.
x2 = x.clone(); x2[1:] = torch.randn(B - 1, d) * 10
print(torch.allclose(batch_norm(x, g, b)[0], batch_norm(x2, g, b)[0]))   # False
print(torch.allclose(layer_norm(x, g, b)[0], layer_norm(x2, g, b)[0]))   # True
